In [1]:
import pandas as pd
import numpy as np

In [2]:
kdd_train= pd.read_csv("kdd_train.csv")

In [3]:
kdd_test= pd.read_csv("kdd_test.csv")

In [4]:
kdd_train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack,last_flag,label_5,label_2
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,Normal,normal
1,0,udp,other,SF,146,0,0,0,0,0,...,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,Normal,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,DoS,attack
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,Normal,normal
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,Normal,normal


In [5]:
kdd_test.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack,last_flag,label_5,label_2
0,0,tcp,private,REJ,0,0,0,0,0,0,...,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21,DoS,attack
1,0,tcp,private,REJ,0,0,0,0,0,0,...,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21,DoS,attack
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.61,0.02,0.0,0.0,0.00,0.00,normal,21,Normal,normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,1.00,0.28,0.0,0.0,0.00,0.00,saint,15,Probe,attack
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.03,0.02,0.0,0.0,0.83,0.71,mscan,11,Probe,attack


In [6]:
print("kdd_train:",kdd_train.shape)

kdd_train: (125973, 45)


In [7]:
print("kdd_test:",kdd_test.shape)

kdd_test: (22544, 45)


#### Importing categorical columns list

In [8]:
categ_columns=['protocol_type', 'service', 'flag']

In [9]:
print(categ_columns)

['protocol_type', 'service', 'flag']


In [10]:
kdd_train[categ_columns].head()

,protocol_type,service,flag
0,tcp,ftp_data,SF
1,udp,other,SF
2,tcp,private,S0
3,tcp,http,SF
4,tcp,http,SF


In [11]:
kdd_test[categ_columns].head()

,protocol_type,service,flag
0,tcp,private,REJ
1,tcp,private,REJ
2,tcp,ftp_data,SF
3,icmp,eco_i,SF
4,tcp,telnet,RSTO


In [12]:
##Saving labels to a dataframe for reference
label_list=[ 'attack','label_2','label_5']
kdd_train_label=kdd_train[label_list]
print(kdd_train_label.head())
print("kdd_train_label:", kdd_train_label.shape)

    attack label_2 label_5
0   normal  normal  Normal
1   normal  normal  Normal
2  neptune  attack     DoS
3   normal  normal  Normal
4   normal  normal  Normal
kdd_train_label: (125973, 3)


In [13]:
kdd_test_label=kdd_test[label_list]
print(kdd_test_label.head())
print("kdd_test_label:", kdd_test_label.shape)

    attack label_2 label_5
0  neptune  attack     DoS
1  neptune  attack     DoS
2   normal  normal  Normal
3    saint  attack   Probe
4    mscan  attack   Probe
kdd_test_label: (22544, 3)


In [14]:
#su_attempted takes only values 0 and 1
kdd_train['su_attempted'].replace(2,0,inplace=True)
#su_attempted takes only values 0 and 1
kdd_test['su_attempted'].replace(2,0,inplace=True)
#num_file_creations shows an abnormal value 100.Repalcing that with
#majority value 0
kdd_test['num_file_creations'].replace(100,0,inplace=True)

In [15]:
insig_list=['num_outbound_cmds','is_host_login']
corr_drop_list=['num_root','srv_serror_rate','srv_rerror_rate','dst_host_srv_serror_rate','dst_host_srv_rerror_rate',
             'dst_host_serror_rate','dst_host_rerror_rate','dst_host_same_srv_rate']


In [16]:
##Columns to be dropped after Encoding
to_drop_list=  label_list +insig_list
print(to_drop_list)

['attack', 'label_2', 'label_5', 'num_outbound_cmds', 'is_host_login']


### ONE HOT ENCODING

from sklearn.preprocessing import OneHotEncoder


enc = OneHotEncoder(handle_unknown='ignore')

train_enc = pd.DataFrame(enc.fit_transform(kdd_train[categ_columns]).toarray())
kdd_train_enc = kdd_train.join(train_enc)
kdd_train_enc.head()

test_enc = pd.DataFrame(enc.fit_transform(kdd_test[categ_columns]).toarray())
kdd_test_enc = kdd_test.join(train_enc)
kdd_test_enc.head()

kdd_train_enc.drop(to_drop_list, axis = 1, inplace = True)

kdd_test_enc.drop(to_drop_list, axis = 1, inplace = True)

kdd_train_enc.describe()

kdd_test_enc.describe()

### Dummy variable Encoding

In [17]:
#Selecting columns to create dummy variables
dummy_Var_list= categ_columns
print(dummy_Var_list)

['protocol_type', 'service', 'flag']


In [18]:
# Taking the data for categorical variables into a dictionary 

Dict_values={}
for col_name in dummy_Var_list:

        Dict_values[col_name]=kdd_train[col_name].unique()
    
string=""    
    
for key, value in Dict_values.items():
    string+=str(key) + ":" +str(value) + "\n"

    
print(string)


protocol_type:['tcp' 'udp' 'icmp']
service:['ftp_data' 'other' 'private' 'http' 'remote_job' 'name' 'netbios_ns'
 'eco_i' 'mtp' 'telnet' 'finger' 'domain_u' 'supdup' 'uucp_path' 'Z39_50'
 'smtp' 'csnet_ns' 'uucp' 'netbios_dgm' 'urp_i' 'auth' 'domain' 'ftp'
 'bgp' 'ldap' 'ecr_i' 'gopher' 'vmnet' 'systat' 'http_443' 'efs' 'whois'
 'imap4' 'iso_tsap' 'echo' 'klogin' 'link' 'sunrpc' 'login' 'kshell'
 'sql_net' 'time' 'hostnames' 'exec' 'ntp_u' 'discard' 'nntp' 'courier'
 'ctf' 'ssh' 'daytime' 'shell' 'netstat' 'pop_3' 'nnsp' 'IRC' 'pop_2'
 'printer' 'tim_i' 'pm_dump' 'red_i' 'netbios_ssn' 'rje' 'X11' 'urh_i'
 'http_8001' 'aol' 'http_2784' 'tftp_u' 'harvest']
flag:['SF' 'S0' 'REJ' 'RSTR' 'SH' 'RSTO' 'S1' 'RSTOS0' 'S3' 'S2' 'OTH']



#### Function to create dummy variable

In [19]:
#Function to craete dummy variables
def dummy_variable(col_name,data):
    #Create Dummies for the variable
    
    temp_dm=pd.get_dummies(data[col_name], drop_first = False)
    
    #Prefixing the dummy columns with appropriate name
    pref_col=str(col_name)+ "_"
    temp_dm=temp_dm.add_prefix(pref_col)
    
    return temp_dm

#### Adding dummy variables dropping one level to the Train dataset

In [20]:
data= kdd_train.copy()
dummy_data= pd.DataFrame()
for col in dummy_Var_list:
    temp_data=dummy_variable(col,data)
       
    #Conacatenating the dummy variables to original dataset
    dummy_data = pd.concat([dummy_data,temp_data], axis = 1)
    
    
dummy_data.head()

,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_IRC,service_X11,service_Z39_50,service_aol,service_auth,service_bgp,service_courier,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [21]:
kdd_train_dm = pd.concat([kdd_train,dummy_data], axis = 1)
     
kdd_train_dm.drop(dummy_Var_list, axis = 1, inplace = True)
        
print("Dummy variables added")

Dummy variables added


In [22]:
kdd_train_dm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Columns: 126 entries, duration to flag_SH
dtypes: float64(15), int64(24), object(3), uint8(84)
memory usage: 50.5+ MB


In [23]:
kdd_train_dm.shape

(125973, 126)

In [24]:
#Drop unwanted columns
kdd_train_dm.drop(label_list, axis = 1, inplace = True)

In [25]:
kdd_train_dm.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
count,125973.00000,1.259730e+05,1.259730e+05,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,...,125973.00000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000
mean,287.14465,4.556674e+04,1.977911e+04,0.000198,0.022687,0.000111,0.204409,0.001222,0.395736,0.279250,...,0.08917,0.012399,0.000818,0.019218,0.276655,0.002897,0.001008,0.000389,0.594929,0.002151
std,2604.51531,5.870331e+06,4.021269e+06,0.014086,0.253530,0.014366,2.149968,0.045239,0.489010,23.942042,...,0.28499,0.110661,0.028583,0.137292,0.447346,0.053750,0.031736,0.019719,0.490908,0.046332
min,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.00000,4.400000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,0.00000,2.760000e+02,5.160000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,42908.00000,1.379964e+09,1.309937e+09,1.000000,3.000000,3.000000,77.000000,5.000000,1.000000,7479.000000,...,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


#### Adding dummy variables dropping one level to the Test dataset

In [26]:
data= kdd_test.copy()
dummy_data= pd.DataFrame()
for col in dummy_Var_list:
    temp_data=dummy_variable(col,data)
       
    #Conacatenating the dummy variables to original dataset
    dummy_data = pd.concat([dummy_data,temp_data], axis = 1)
    
dummy_data.head()    
    

,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_IRC,service_X11,service_Z39_50,service_auth,service_bgp,service_courier,service_csnet_ns,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [27]:
kdd_test_dm = pd.concat([kdd_test,dummy_data], axis = 1)
     
kdd_test_dm.drop(dummy_Var_list, axis = 1, inplace = True)
        
print("Dummy variables added")

Dummy variables added


In [28]:
kdd_test_dm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Columns: 120 entries, duration to flag_SH
dtypes: float64(15), int64(24), object(3), uint8(78)
memory usage: 8.9+ MB


In [29]:
kdd_test_dm.shape

(22544, 120)

In [30]:
#Drop unwanted columns
kdd_test_dm.drop(label_list, axis = 1, inplace = True)

In [31]:
kdd_test_dm.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
count,22544.000000,2.254400e+04,2.254400e+04,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,...,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000
mean,218.859076,1.039545e+04,2.056019e+03,0.000311,0.008428,0.000710,0.105394,0.021647,0.442202,0.119899,...,0.170777,0.034289,0.000089,0.029675,0.089292,0.000932,0.000665,0.011045,0.659821,0.003238
std,1407.176612,4.727864e+05,2.121930e+04,0.017619,0.142599,0.036473,0.928428,0.150328,0.496659,7.269597,...,0.376322,0.181973,0.009419,0.169694,0.285171,0.030507,0.025787,0.104516,0.473780,0.056813
min,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,5.400000e+01,4.600000e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,0.000000,2.870000e+02,6.010000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,57715.000000,6.282565e+07,1.345927e+06,1.000000,3.000000,3.000000,101.000000,4.000000,1.000000,796.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Normalization


## Minmax scaling of Train and test dataset

In [32]:
kdd_train_dm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Columns: 123 entries, duration to flag_SH
dtypes: float64(15), int64(24), uint8(84)
memory usage: 47.6 MB


In [33]:
kdd_test_dm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Columns: 117 entries, duration to flag_SH
dtypes: float64(15), int64(24), uint8(78)
memory usage: 8.4 MB


In [34]:
col_list_train=kdd_train_dm.columns

In [35]:
col_list_test=kdd_test_dm.columns

In [36]:
#Importing Minmax scaler 
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

In [37]:
kdd_train_norm=scaler.fit_transform(kdd_train_dm)
kdd_train_norm= pd.DataFrame(kdd_train_norm, columns= col_list_train)

In [38]:
kdd_test_norm=scaler.fit_transform(kdd_test_dm)
kdd_test_norm= pd.DataFrame(kdd_test_norm, columns= col_list_test)

In [39]:
kdd_train_norm.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
count,125973.000000,1.259730e+05,1.259730e+05,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,...,125973.00000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000
mean,0.006692,3.302024e-05,1.509928e-05,0.000198,0.007562,0.000037,0.002655,0.000244,0.395736,0.000037,...,0.08917,0.012399,0.000818,0.019218,0.276655,0.002897,0.001008,0.000389,0.594929,0.002151
std,0.060700,4.253974e-03,3.069818e-03,0.014086,0.084510,0.004789,0.027922,0.009048,0.489010,0.003201,...,0.28499,0.110661,0.028583,0.137292,0.447346,0.053750,0.031736,0.019719,0.490908,0.046332
min,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,3.188489e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,0.000000,2.000052e-07,3.939120e-07,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000e+00,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [40]:
kdd_test_norm.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
count,22544.000000,2.254400e+04,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,...,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000
mean,0.003792,1.654651e-04,0.001528,0.000311,0.002809,0.000237,0.001044,0.005412,0.442202,0.000151,...,0.170777,0.034289,0.000089,0.029675,0.089292,0.000932,0.000665,0.011045,0.659821,0.003238
std,0.024381,7.525373e-03,0.015766,0.017619,0.047533,0.012158,0.009192,0.037582,0.496659,0.009133,...,0.376322,0.181973,0.009419,0.169694,0.285171,0.030507,0.025787,0.104516,0.473780,0.056813
min,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,8.595216e-07,0.000034,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,0.000000,4.568198e-06,0.000447,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [41]:
kdd_train_norm = pd.concat([kdd_train_norm,kdd_train_label], axis = 1)
kdd_train_norm.head()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH,attack,label_2,label_5
0,0.0,3.558064e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal
1,0.0,1.057999e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal
2,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,neptune,attack,DoS
3,0.0,1.681203e-07,6.223962e-06,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal
4,0.0,1.442067e-07,3.206260e-07,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal


In [42]:
kdd_test_norm = pd.concat([kdd_test_norm,kdd_test_label], axis = 1)
kdd_test_norm.head()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH,attack,label_2,label_5
0,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,neptune,attack,DoS
1,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,neptune,attack,DoS
2,0.000035,2.066513e-04,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,normal,normal,Normal
3,0.000000,3.183413e-07,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,saint,attack,Probe
4,0.000017,0.000000e+00,0.000011,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,mscan,attack,Probe


In [43]:
print("kdd_train_norm:",kdd_train_norm.shape)
print("kdd_test_norm:",kdd_test_norm.shape)

kdd_train_norm: (125973, 126)
kdd_test_norm: (22544, 120)


In [44]:
kdd_train_norm.to_csv("kdd_train_raw.csv",index=False)

In [45]:
kdd_test_norm.to_csv("kdd_test_raw.csv",index=False)